# QUBO benchmark: single A100 cluster, 15-minute scheduling

This notebook evaluates the reduced QUBO on the deliberately difficult one-cluster instances generated in notebook 20. It has four explicit goals:

1. Quantify logical QUBO size and graph connectivity before attempting sampling.
2. Calibrate penalty weights on development instances, never on the held-out target.
3. Compare schedules using a common physical post-evaluator.
4. Keep the QUBO proxy objective separate from the full MILP objective.

The QUBO contains fixed-PUE effective-price energy cost, exact-one assignment, GPU capacity with slack bits, and optional squared-load smoothing. It does not yet contain explicit renewable/grid dispatch, battery dynamics, or the exact contracted-grid-peak term. Those are evaluated after decoding a schedule.

## Experiment protocol

- **Development set:** `long_wide_window` seeds 7601 and 7603. These select penalty ranges.
- **Held-out target:** seed 7602. Gurobi reached its 60-second limit on this instance with a feasible incumbent and a nonzero optimality gap.
- **Proxy comparison:** an energy-only MILP uses the same effective price as the QUBO. This isolates the impact of the optimization method from the renewable-dispatch approximation.
- **Physical comparison:** any feasible decoded schedule is re-evaluated with actual renewable availability, grid prices, PUE, grid import, and optional contracted-power charge.

A valid QUBO schedule must assign every job once, start within its window, and respect the 256-node capacity at every 15-minute slot. Slack-bit consistency is monitored separately: it diagnoses the penalty encoding but does not replace physical schedule feasibility.

In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dwave.samplers import SimulatedAnnealingSampler

from src.config import ModelConfig
from src.milp.gurobi_model import build_milp_model
from src.milp.solve import solve_model
from src.quantum.benchmarking import evaluate_physical_schedule, make_effective_price_qubo_inputs
from src.quantum.qubo_builder import (
    QuboPenaltyWeights,
    build_scheduling_qubo,
    decode_qubo_sample,
    summarize_qubo_graph,
    to_dimod_bqm,
    validate_decoded_schedule,
)


In [2]:
REPOSITORY_ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
INSTANCE_ROOT = REPOSITORY_ROOT / 'data/instances/single_cluster_15m/long_duration_focused'
DEVELOPMENT_SEEDS = (7601, 7603)
TARGET_SEED = 7602
PROFILE_NAME = 'long_wide_window'
DELTA_T = 0.25
CONTRACTED_POWER_RATIO = 0.70
PEAK_PRICE = 1_000.0

# Keep long experiments opt-in. The profiling cells above them are quick.
RUN_CALIBRATION = False
RUN_TARGET_QUBO = False
RUN_PROXY_MILP = False
NUM_READS = 150
RANDOM_SEED = 41

# Capacity is normalized by its 256-node scale through the grid below.
# This range is deliberately screened on development instances only.
ASSIGNMENT_GRID = (1_000.0, 3_000.0, 10_000.0)
GPU_CAPACITY_GRID = (0.001, 0.01, 0.05, 0.10)
PEAK_GRID = (0.0, 0.01, 0.10)


In [3]:
def load_instance(seed: int):
    """Load one persisted 15-minute one-cluster stress instance."""
    instance_dir = INSTANCE_ROOT / f'{PROFILE_NAME}_seed_{seed}'
    jobs = pd.read_csv(instance_dir / 'jobs.csv')
    clusters = pd.read_csv(instance_dir / 'clusters.csv')
    energy = pd.read_csv(instance_dir / 'energy_inputs_15m.csv')
    gurobi = pd.read_csv(instance_dir / 'gurobi_summary.csv')
    return instance_dir, jobs, clusters, energy, gurobi

def proxy_model_config(clusters: pd.DataFrame, energy: pd.DataFrame) -> ModelConfig:
    """Return an energy-only MILP configuration aligned with QUBO prices."""
    return ModelConfig(
        contracted_power=float(clusters['capacity'].sum()) * 10.0,
        renewable_price=1_000_000.0,
        peak_price=0.0,
        delta_t=DELTA_T,
        pue=float(energy['pue'].iloc[0]),
    )

def extract_milp_schedule(model, variables, jobs: pd.DataFrame) -> pd.DataFrame:
    """Decode selected MILP assignment variables into the common schedule schema."""
    rows = []
    jobs_by_id = jobs.set_index('job_id')
    for (job_id, cluster, start), variable in variables['x'].items():
        if variable.X <= 0.5:
            continue
        job = jobs_by_id.loc[job_id]
        rows.append({
            'job_id': job_id, 'assigned_cluster': cluster, 'start_hour': int(start),
            'duration': int(job.duration), 'power': float(job.power),
            'gpu_count_required': float(job.gpu_count_required),
        })
    return pd.DataFrame(rows).sort_values(['start_hour', 'job_id']).reset_index(drop=True) if rows else pd.DataFrame(rows)


In [4]:
target_dir, target_jobs, target_clusters, target_energy, target_gurobi = load_instance(TARGET_SEED)
target_qubo_energy = make_effective_price_qubo_inputs(target_energy)
contracted_power_mw = CONTRACTED_POWER_RATIO * float(target_clusters['capacity'].sum()) * float(target_energy['pue'].iloc[0])

print(f'Target: {target_dir.name}')
print(f'Jobs: {len(target_jobs)} | slots: {len(target_energy)} | node capacity: {int(target_clusters.gpu_count.iloc[0])}')
print(f'Contracted grid power: {contracted_power_mw:.4f} MW | slot length: {DELTA_T:.2f} h')
display(target_gurobi)


Target: long_wide_window_seed_7602
Jobs: 42 | slots: 96 | node capacity: 256
Contracted grid power: 0.7168 MW | slot length: 0.25 h


,instance,profile,seed,status,has_solution,runtime_s,mip_gap,objective,best_bound,assignment_variables,constraints,contracted_power_ratio,utilization,mean_duration_slots,mean_node_demand,mean_start_options,mean_price_spread_ratio,cheap_peak_potential_load_ratio,hidden_peak_load_ratio,config
0,long_wide_window_seed_7602,long_wide_window,7602,9,True,60.007171,0.010206,2173.683245,2151.499402,703,523,0.7,0.820312,9.142857,52.190476,16.738095,1.413874,3.6875,1.0,"{""num_jobs"": 42, ""node_capacity"": 256, ""slot_m..."


## Logical QUBO profile

Logical variables are not physical QPU qubits. The count includes assignment variables and binary slack bits. The coupling count and degree are important because dense logical graphs make minor embedding on an annealer substantially more difficult.

In [5]:
profile_weights = QuboPenaltyWeights(assignment=1_000.0, gpu_capacity=0.01, peak_smoothing=0.01)
target_qubo = build_scheduling_qubo(
    target_jobs, target_qubo_energy, target_clusters,
    proxy_model_config(target_clusters, target_energy), profile_weights,
)
qubo_graph = summarize_qubo_graph(target_qubo)
display(pd.DataFrame([qubo_graph]))
display(pd.DataFrame(target_qubo.metadata['terms']).T)


,logical_variables,assignment_variables,slack_variables,linear_terms,quadratic_couplers,coupling_density,mean_degree,max_degree
0,1567,703,864,1567,113091,0.092172,144.340779,391


,linear_terms_after_term,quadratic_terms_after_term,description
energy_fixed_pue,703,0,Linear fixed-PUE energy cost using effective t...
assignment,703,5640,Exact one-hot job assignment penalty.
gpu_capacity_with_slack,1567,113091,Exact GPU capacity penalty using binary unused...
peak_fixed_pue,1567,113091,Quadratic fixed-PUE facility-load smoothing pr...


## Common physical evaluator

The following evaluator is used after either solver emits a feasible schedule. It applies the actual solar profile first when it is cheaper than the grid, then charges the remaining grid import. Peak is the maximum **grid import**, as in the current MILP. This gives a fair operational comparison even though the QUBO was optimized with effective prices.

In [6]:
def evaluate_schedule(schedule: pd.DataFrame, energy: pd.DataFrame, clusters: pd.DataFrame):
    """Return physical profile and summary for a decoded feasible schedule."""
    contracted = CONTRACTED_POWER_RATIO * float(clusters['capacity'].sum()) * float(energy['pue'].iloc[0])
    return evaluate_physical_schedule(
        schedule, energy, delta_t=DELTA_T, contracted_power_mw=contracted, peak_price=PEAK_PRICE
    )

def plot_schedule_energy(schedule: pd.DataFrame, energy: pd.DataFrame, clusters: pd.DataFrame, title: str):
    """Plot decoded facility load against renewable supply and QUBO price."""
    profile, _ = evaluate_schedule(schedule, energy, clusters)
    figure, left = plt.subplots(figsize=(12, 4))
    left.step(profile['hour'], profile['facility_load_mw'], where='mid', label='Facility load (MW)', color='#1f77b4')
    left.step(profile['hour'], profile['renewable_available_mw'], where='mid', label='Renewable available (MW)', color='#2ca02c')
    left.step(profile['hour'], profile['grid_used_mw'], where='mid', label='Grid import (MW)', color='#d62728')
    left.set_xlabel('15-minute slot'); left.set_ylabel('Power (MW)')
    right = left.twinx()
    right.plot(energy['hour'], energy['effective_price'], color='#333333', alpha=0.6, label='QUBO effective price')
    right.set_ylabel('Price')
    handles, labels = left.get_legend_handles_labels(); handles2, labels2 = right.get_legend_handles_labels()
    left.legend(handles + handles2, labels + labels2, loc='upper right')
    left.set_title(title); plt.show()


## QUBO sampling and feasibility selection

Simulated annealing is used here as a local, reproducible sampler. Every returned sample is decoded and checked. We select the lowest-QUBO-energy **physically feasible** sample; if no sample is feasible, the result reports the best sampled assignment violation instead of silently comparing an invalid schedule to Gurobi.

In [7]:
def assignment_error_count(schedule: pd.DataFrame, jobs: pd.DataFrame) -> int:
    """Count missing jobs plus extra assignments in a decoded schedule."""
    counts = schedule['job_id'].value_counts() if not schedule.empty else pd.Series(dtype=int)
    return sum(abs(int(counts.get(job_id, 0)) - 1) for job_id in jobs['job_id'])

def sample_qubo(jobs, clusters, energy, weights, *, num_reads, seed):
    """Sample one QUBO and return the best feasible decoded schedule, if any."""
    qubo_inputs = make_effective_price_qubo_inputs(energy)
    qubo = build_scheduling_qubo(jobs, qubo_inputs, clusters, proxy_model_config(clusters, energy), weights)
    start = perf_counter()
    sampleset = SimulatedAnnealingSampler().sample(to_dimod_bqm(qubo), num_reads=num_reads, seed=seed)
    runtime_s = perf_counter() - start
    hours = energy['hour'].astype(int).tolist()
    decoded = []
    for record in sampleset.data(fields=['sample', 'energy']):
        schedule = decode_qubo_sample(qubo, dict(record.sample))
        validation = validate_decoded_schedule(schedule, jobs, clusters, hours)
        decoded.append({
            'qubo_energy': float(record.energy), 'schedule': schedule, 'validation': validation,
            'assignment_error_count': assignment_error_count(schedule, jobs),
        })
    feasible = [row for row in decoded if row['validation']['feasible']]
    selected = min(feasible or decoded, key=lambda row: (not row['validation']['feasible'], row['assignment_error_count'], row['qubo_energy']))
    physical_profile, physical_summary = evaluate_schedule(selected['schedule'], energy, clusters)
    return {
        'qubo': qubo, 'schedule': selected['schedule'], 'physical_profile': physical_profile,
        'physical_summary': physical_summary, 'selected_qubo_energy': selected['qubo_energy'],
        'selected_validation': selected['validation'], 'feasible_sample_count': len(feasible),
        'num_reads': num_reads, 'runtime_s': runtime_s, 'best_assignment_error_count': selected['assignment_error_count'],
    }


In [8]:
def solve_proxy_milp(jobs, clusters, energy, *, time_limit=60):
    """Solve the energy-only MILP that uses exactly the QUBO effective prices."""
    qubo_inputs = make_effective_price_qubo_inputs(energy)
    model, variables = build_milp_model(
        jobs, qubo_inputs, clusters, proxy_model_config(clusters, energy),
        model_name='qubo_aligned_proxy_milp',
        enforce_gpu_constraints=True, enforce_cpu_constraints=False, enforce_memory_constraints=False,
    )
    model.Params.OutputFlag = 0
    start = perf_counter(); solve_model(model, time_limit=time_limit, mip_gap=0.001); runtime_s = perf_counter() - start
    schedule = extract_milp_schedule(model, variables, jobs) if model.SolCount else pd.DataFrame()
    validation = validate_decoded_schedule(schedule, jobs, clusters, energy['hour'].astype(int).tolist()) if model.SolCount else {'feasible': False, 'violations': ['no incumbent']}
    profile, summary = evaluate_schedule(schedule, energy, clusters)
    return {'schedule': schedule, 'validation': validation, 'physical_profile': profile, 'physical_summary': summary, 'runtime_s': runtime_s, 'objective': float(model.ObjVal) if model.SolCount else None, 'mip_gap': float(model.MIPGap) if model.SolCount else None}


## Development-only penalty calibration

The grid is intentionally modest. It screens orders of magnitude, then the winning region can be refined. Rank configurations lexicographically: number of feasible samples, assignment errors, QUBO energy, then physical cost. The held-out target is not used here.

In [9]:
def calibrate_penalties():
    """Screen penalty weights on development seeds and return ranked rows."""
    rows = []
    for seed in DEVELOPMENT_SEEDS:
        _, jobs, clusters, energy, _ = load_instance(seed)
        for assignment in ASSIGNMENT_GRID:
            for gpu_capacity in GPU_CAPACITY_GRID:
                for peak in PEAK_GRID:
                    result = sample_qubo(jobs, clusters, energy, QuboPenaltyWeights(assignment, gpu_capacity, peak), num_reads=NUM_READS, seed=RANDOM_SEED + seed)
                    rows.append({
                        'seed': seed, 'assignment': assignment, 'gpu_capacity': gpu_capacity, 'peak': peak,
                        'feasible_samples': result['feasible_sample_count'],
                        'assignment_errors': result['best_assignment_error_count'],
                        'qubo_energy': result['selected_qubo_energy'],
                        'physical_total_cost': result['physical_summary']['physical_total_cost'],
                        'runtime_s': result['runtime_s'],
                    })
    result = pd.DataFrame(rows)
    return result.sort_values(['feasible_samples', 'assignment_errors', 'qubo_energy'], ascending=[False, True, True])

if RUN_CALIBRATION:
    calibration_results = calibrate_penalties()
    display(calibration_results.head(15))


## Held-out target evaluation

First run the calibrated QUBO weights on seed 7602. Then optionally run the proxy-aligned MILP. The table compares only valid schedules; an infeasible QUBO sample remains a diagnostic result, not an energy-cost competitor.

In [10]:
# Replace this with the best development-only calibration result.
SELECTED_WEIGHTS = QuboPenaltyWeights(assignment=1_000.0, gpu_capacity=0.01, peak_smoothing=0.01)

if RUN_TARGET_QUBO:
    target_qubo_result = sample_qubo(target_jobs, target_clusters, target_energy, SELECTED_WEIGHTS, num_reads=NUM_READS, seed=RANDOM_SEED)
    print(target_qubo_result['selected_validation'])
    print({'feasible_samples': target_qubo_result['feasible_sample_count'], **target_qubo_result['physical_summary']})
    if target_qubo_result['selected_validation']['feasible']:
        plot_schedule_energy(target_qubo_result['schedule'], target_energy, target_clusters, 'QUBO target schedule')

if RUN_PROXY_MILP:
    target_proxy_milp = solve_proxy_milp(target_jobs, target_clusters, target_energy)
    print(target_proxy_milp['validation'])
    print({'objective': target_proxy_milp['objective'], 'mip_gap': target_proxy_milp['mip_gap'], **target_proxy_milp['physical_summary']})
    if target_proxy_milp['validation']['feasible']:
        plot_schedule_energy(target_proxy_milp['schedule'], target_energy, target_clusters, 'Proxy-aligned MILP target schedule')


## Interpretation for the benchmark

- A small QUBO energy is not enough: only decoded schedules passing assignment and GPU checks are comparable.
- A feasible QUBO schedule with a higher physical cost than the proxy MILP is a valid heuristic result, not an optimality result.
- If Gurobi reaches its time limit with an incumbent and gap, report incumbent, bound, gap, and time. Do not call that an unsolved instance.
- Small instances can validate the encoding against known optima. The 15-minute target tests scaling and feasibility. Neither by itself demonstrates quantum advantage.
- Direct QPU execution is considered only after logical size, coupling density, and an embedding feasibility check are recorded.